# Pancancer enrichment analysis step 11: GSEApy PreRank with 1 - p

## Setup

In [1]:
import pandas as pd
import numpy as np
import os
import datetime
import gseapy as gp

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
NUM_PERMUTATIONS = 1000

STEP1_DIR = "step1_outputs"
STEP1_FILE = "tumor_change_20200529_195104_10000_perm.tsv"
STEP1_FILE_PATH = os.path.join(STEP1_DIR, STEP1_FILE)

STEP11_DIR = "step11_outputs"

# Create log dir and file
LOG_DIR = "step11_checkpoints"
LOG_FILE = f"{TIME_START}_prerank.log"
LOG_FILE_PATH = os.path.join(LOG_DIR, LOG_FILE)

if not os.path.isdir(LOG_DIR):
    os.mkdir(LOG_DIR)
    
with open(LOG_FILE_PATH, 'w') as fp: 
    fp.write(f"{TIME_START}\n")
    fp.write(f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Started step 11 with {NUM_PERMUTATIONS} permutations.\n")

GSEAPY_DIR_PATH = os.path.join(STEP11_DIR, "gseapy")

if not os.path.isdir(STEP11_DIR):
    os.mkdir(STEP11_DIR)

## Prepare data

In [3]:
# Read in the file from step 1
data = pd.read_csv(STEP1_FILE_PATH, sep='\t', index_col=0)
data = data[["cancer_type", "protein_str", "adj_p"]]

In [4]:
# Create 1 - adj_p column
data = data.assign(one_minus_adj_p=1 - data["adj_p"])[["cancer_type", "protein_str", "one_minus_adj_p"]]

In [5]:
data

,cancer_type,protein_str,one_minus_adj_p
0,ccrcc,A1BG,1.000000
1,ccrcc,A1CF,1.000000
2,ccrcc,A2M,1.000000
3,ccrcc,A4GALT,0.785690
4,ccrcc,AAAS,1.000000
...,...,...,...
86254,ovarian,ZWINT,0.147564
86255,ovarian,ZYG11B,0.741230
86256,ovarian,ZYX,1.000000
86257,ovarian,ZZEF1,0.999812


## Run GSEApy PreRank analysis

In [6]:
def gseapy_prerank_reactome(ranks, asc, wst):
    
    with open(LOG_FILE_PATH, 'a') as fp: 
        fp.write(f"\n{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Started asc={asc} wst={wst}.\n")
    
    output_file = f"enrichment_gseapy_prerank_1map_{TIME_START}_{NUM_PERMUTATIONS}_perms_asc_{asc}_wst_{wst}.tsv"
    output_path = os.path.join(STEP11_DIR, output_file)
    
    # For each cancer, find enriched pathways.
    all_enrichments = pd.DataFrame()

    for cancer_type in ranks["cancer_type"].unique():

        prot = ranks.loc[
            ranks["cancer_type"] == cancer_type,
            ["protein_str", "one_minus_adj_p"]
        ]
        
        pre_res = gp.prerank(
            rnk=prot,
            gene_sets="gene_set_libraries/Reactome_2020.gmt",
            outdir=GSEAPY_DIR_PATH,
            permutation_num=NUM_PERMUTATIONS,
            min_size=1,
            max_size=500, 
            weighted_score_type=wst,
            ascending=asc,
            no_plot=True,
            processes=1,
            seed=0)

        cancer_enriched = pre_res.res2d.assign(cancer_type=cancer_type)
        all_enrichments = all_enrichments.append(cancer_enriched)

        # Log that we finished this cancer type
        with open(LOG_FILE_PATH, 'a') as fp: 
            fp.write(f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Finished {cancer_type} data.\n")
            
    # Record the results
    all_enrichments.to_csv(output_path, sep="\t")
            
    with open(LOG_FILE_PATH, 'a') as fp: 
        fp.write(f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Finished asc={asc} wst={wst}.\n")

In [7]:
gseapy_prerank_reactome(data, asc=False, wst=1)

2020-06-19 11:12:36,697 Input gene rankings contains duplicated IDs, Only use the duplicated ID with highest value!
/fslhome/lindgcm4/anaconda3/envs/dev/lib/python3.7/site-packages/gseapy/algorithm.py:73: RuntimeWarning: divide by zero encountered in true_divide
  norm_tag =  1.0/sum_correl_tag
/fslhome/lindgcm4/anaconda3/envs/dev/lib/python3.7/site-packages/gseapy/algorithm.py:76: RuntimeWarning: invalid value encountered in multiply
  RES = np.cumsum(tag_indicator * correl_vector * norm_tag - no_tag_indicator * norm_no_tag, axis=axis)
/fslhome/lindgcm4/anaconda3/envs/dev/lib/python3.7/site-packages/gseapy/algorithm.py:83: RuntimeWarning: invalid value encountered in greater
  es_vec = np.where(np.abs(max_ES) > np.abs(min_ES), max_ES, min_ES)
2020-06-19 12:41:50,557 Input gene rankings contains duplicated IDs, Only use the duplicated ID with highest value!
2020-06-19 13:05:58,584 Input gene rankings contains duplicated IDs, Only use the duplicated ID with highest value!
2020-06-19 13: